In [18]:
import os
import sys
import yaml
import pandas as pd
import numpy as np

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import stars_p, parse_casenum

from matplotlib import pyplot as plt
from scipy.stats import f_oneway, chi2_contingency

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10

In [19]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_distances_suffixes.parquet"))
reasons_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "reasons.parquet"))
sfx_df = pd.read_csv(os.path.join(LOCAL_PATH, "intermediate_data/cpc", "suffix_groups.csv"))


In [20]:
reasons_agg_df = reasons_df.groupby('reason').agg(count=('item_no','count')).sort_values(by='count', ascending=False).reset_index()

In [21]:
SFX_NAME_MAP = {}
for _, row in sfx_df.iterrows():
    SFX_NAME_MAP[row['category_code']] = row['category']

In [22]:
df['outcome'] = df['project_result'].map({
    'APPROVED': 'Approved',
    'APPROVED WITH MINOR CONDITIONS OR MODIFICATIONS': 'Approved w/ conditions',
    'APPROVED WITH MAJOR CONDITIONS OR MODIFICATIONS': 'Approved w/ conditions',
    'DENIED': 'Delayed / Denied',
    'APPLICATION WITHDRAWN': 'Delayed / Denied',
    'DELAYED': 'Delayed / Denied'
})
n_approved = f"{(df['outcome'] == "Approved").sum():,.0f}"
n_approved_w_conditions = f"{(df['outcome'] == "Approved w/ conditions").sum():,.0f}"
n_delayed_denied = f"{(df['outcome'] == "Delayed / Denied").sum():,.0f}"

In [23]:
def get_descriptive_stats_continuous(variable):
    # Get descriptive stats for continuous variables
    approved_mask = df['outcome'] == "Approved"
    approved_w_conditions_mask = df['outcome'] == "Approved w/ conditions"
    delayed_denied_mask = df['outcome'] == "Delayed / Denied"
    
    # Calculate mean and standard deviation for approved cases
    approved_mean = df.loc[approved_mask, variable].mean()
    approved_std = df.loc[approved_mask, variable].std()
    
    # Calculate mean and standard deviation for approved w/ conditions cases
    approved_w_conditions_mean = df.loc[approved_w_conditions_mask, variable].mean()
    approved_w_conditions_std = df.loc[approved_w_conditions_mask, variable].std()

    # Calculate mean and standard deviation for delayed/denied cases
    delayed_denied_mean = df.loc[delayed_denied_mask, variable].mean()
    delayed_denied_std = df.loc[delayed_denied_mask, variable].std()
    
    # Calculate F-test for mean differences between groups
    f_stat, p_value = f_oneway(
        df.loc[approved_mask, variable].dropna(),
        df.loc[approved_w_conditions_mask, variable].dropna(),
        df.loc[delayed_denied_mask, variable].dropna()
    )
    
    return approved_mean, approved_w_conditions_mean, delayed_denied_mean, f_stat, p_value

In [24]:
def get_descriptive_stats_boolean(variable):
    # Get descriptive stats for boolean variables
    approved_mask = df['outcome'] == "Approved"
    approved_w_conditions_mask = df['outcome'] == "Approved w/ conditions"
    delayed_denied_mask = df['outcome'] == "Delayed / Denied"
    
    # Calculate mean for approved cases
    approved_mean = df.loc[approved_mask, variable].mean()

    # Calculate mean for approved w/ conditions cases
    approved_w_conditions_mean = df.loc[approved_w_conditions_mask, variable].mean()

    # Calculate mean for delayed/denied cases
    delayed_denied_mean = df.loc[delayed_denied_mask, variable].mean()

    # Calculate chi-square test for independence between groups
    contingency_table = pd.crosstab(df['outcome'], df[variable])
    chi2, p, dof, expected = chi2_contingency(contingency_table)    
    
    return approved_mean, approved_w_conditions_mean, delayed_denied_mean, chi2, p

In [25]:
header = fr"""
\begin{{table}}[H]
\centering
\caption{{Descriptive Statistics by Motion Outcome---Public Input}}
\vspace{{0.2cm}}
\label{{tab_bivariate_descriptives_public_input}}
\begin{{adjustbox}}{{max width=\textwidth}}
\begin{{threeparttable}}
\begin{{tabular}}{{lrrrll}} \toprule
 Variable & \makecell{{Approved \\ ($n = {n_approved}$)}} & \makecell{{Approved w/ \\ conditions \\ ($n = {n_approved_w_conditions}$)}} & \makecell{{Delayed / \\ denied \\ ($n = {n_delayed_denied}$)}} & \makecell{{Test stat}} & \makecell{{p-value}}  \\ \midrule
 & & & & & \\
"""
footer = r"""
 & & & & & \\
\bottomrule
\multicolumn{6}{r}{***: $p < 0.01$, **:$p < 0.05$, *: $p < 0.1$}
\end{tabular}
\begin{tablenotes}
\item {\textit{Notes: } This reports descriptive statistics for variables grouped by motion outcome. To test differences between groups, a one-way ANOVA F-test was conducted for continuous variables and a chi-square test was conducted for categorical variables.}
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
tbl = ""

var = "n_docs"
varname = "Number of letters (mean)"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:.2f} & {results[1]:.2f} & {results[2]:.2f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

tbl += r" & & & & & \\" + "\n"

var = "n_support"
varname = "~ ~ in support"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:.2f} & {results[1]:.2f} & {results[2]:.2f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

tbl += r"~ ~ ~ mentioning \ldots & & & & & \\" + "\n"
for r in reasons_agg_df['reason']:
    mask = (reasons_df['reason']==r) & (reasons_df['support_or_oppose'].isin(['DEFINITELY SUPPORT', 'SOMEWHAT SUPPORT']))
    temp_df = reasons_df.loc[mask].groupby(['date', 'year', 'item_no']).agg(temp = ('reason','count')).reset_index()
    df = df.merge(temp_df, on=['date', 'year', 'item_no'], how='left')
    df['temp'] = df['temp'].fillna(0)
    df['pct_temp'] = (df['temp'] / df['n_docs'].clip(lower=1))*100
    var = "temp"
    varname = f"~ ~ ~ ~ {r}"
    results = get_descriptive_stats_continuous(var)
    #tbl += rf"{varname} & {results[0]:.2f}\% & {results[1]:.2f}\% & {results[2]:.2f}\% & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"
    tbl += rf"{varname} & {results[0]:.2f} & {results[1]:.2f} & {results[2]:.2f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"
    df = df.drop(columns=['temp', 'pct_temp'])

tbl += r" & & & & & \\" + "\n"

var = "n_oppose"
varname = "~ ~ in opposition"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:.2f} & {results[1]:.2f} & {results[2]:.2f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

tbl += r"~ ~ ~ mentioning \ldots & & & & & \\" + "\n"
for r in reasons_agg_df['reason']:
    mask = (reasons_df['reason']==r) & (reasons_df['support_or_oppose'].isin(['DEFINITELY OPPOSE', 'SOMEWHAT OPPOSE']))
    temp_df = reasons_df.loc[mask].groupby(['date', 'year', 'item_no']).agg(temp = ('reason','count')).reset_index()
    df = df.merge(temp_df, on=['date', 'year', 'item_no'], how='left')
    df['temp'] = df['temp'].fillna(0)
    df['pct_temp'] = (df['temp'] / df['n_docs'].clip(lower=1))*100
    var = "temp"
    varname = f"~ ~ ~ ~ {r}"
    results = get_descriptive_stats_continuous(var)
    #tbl += rf"{varname} & {results[0]:.2f}\% & {results[1]:.2f}\% & {results[2]:.2f}\% & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"
    tbl += rf"{varname} & {results[0]:.2f} & {results[1]:.2f} & {results[2]:.2f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"
    df = df.drop(columns=['temp', 'pct_temp'])

print(header + tbl + footer)

with open(os.path.join(LOCAL_PATH, "tables", "tab_bivariate_descriptives_public_input.tex"), "w", encoding='utf-8') as f:
    f.write(header + tbl + footer)




\begin{table}[H]
\centering
\caption{Descriptive Statistics by Motion Outcome---Public Input}
\vspace{0.2cm}
\label{tab_bivariate_descriptives_public_input}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lrrrll} \toprule
 Variable & \makecell{Approved \\ ($n = 354$)} & \makecell{Approved w/ \\ conditions \\ ($n = 328$)} & \makecell{Delayed / \\ denied \\ ($n = 136$)} & \makecell{Test stat} & \makecell{p-value}  \\ \midrule
 & & & & & \\
Number of letters (mean) & 4.41 & 10.04 & 11.12 & $F = 6.45$ & $0.002^{***}$ \\
 & & & & & \\
~ ~ in support & 1.94 & 4.67 & 3.26 & $F = 3.64$ & $0.027^{**}$ \\
~ ~ ~ mentioning \ldots & & & & & \\
~ ~ ~ ~ housing supply / affordability & 1.41 & 2.99 & 1.07 & $F = 2.30$ & $0.101^{}$ \\
~ ~ ~ ~ neighborhood character & 0.42 & 1.24 & 0.76 & $F = 5.80$ & $0.003^{***}$ \\
~ ~ ~ ~ procedural issues & 0.50 & 1.68 & 1.04 & $F = 2.80$ & $0.062^{*}$ \\
~ ~ ~ ~ building height / size / density & 0.32 & 1.23 & 0.65 & $F = 1.16$ & $

In [29]:
header = fr"""
\begin{{table}}[H]
\centering
\caption{{Descriptive Statistics by Motion Outcome---Project Characteristics}}
\vspace{{0.2cm}}
\label{{tab_bivariate_descriptives_project_characteristics}}
\begin{{adjustbox}}{{max width=\textwidth}}
\begin{{threeparttable}}
\begin{{tabular}}{{lrrrll}} \toprule
 Variable & \makecell{{Approved \\ ($n = {n_approved}$)}} & \makecell{{Approved w/ \\ conditions \\ ($n = {n_approved_w_conditions}$)}} & \makecell{{Delayed / \\ denied \\ ($n = {n_delayed_denied}$)}} & \makecell{{Test stat}} & \makecell{{p-value}}  \\ \midrule
 & & & & & \\
"""
footer = r"""
 & & & & & \\
\bottomrule
\multicolumn{6}{r}{***: $p < 0.01$, **:$p < 0.05$, *: $p < 0.1$}
\end{tabular}
\begin{tablenotes}
\item {\textit{Notes: } This reports descriptive statistics for variables grouped by motion outcome. To test differences between groups, a one-way ANOVA F-test was conducted for continuous variables and a chi-square test was conducted for categorical variables.}
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
tbl = ""

tbl += r"\textit{Case type} & & & & & \\" + "\n"

df['is_residential'] = df['project_type'].isin(['RESIDENTIAL DEVELOPMENT', 'PERMANENT SUPPORTIVE HOUSING / HOMELESS SHELTER', 'SENIOR CARE / ASSISTED LIVING FACILITY'])
var = "is_residential"
varname = r"~ ~ Residential development (\%)"
results = get_descriptive_stats_boolean(var)
tbl += rf"{varname} & {results[0]:.3f} & {results[1]:.3f} & {results[2]:.3f} & $\chi^2 = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

df['is_mixed_use'] = df['project_type'].isin(['MIXED-USE DEVELOPMENT'])
var = 'is_mixed_use'
varname = r"~ ~ Mixed use development (\%)"
results = get_descriptive_stats_boolean(var)
tbl += rf"{varname} & {results[0]:.3f} & {results[1]:.3f} & {results[2]:.3f} & $\chi^2 = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

df['is_nonresidential'] = df['project_type'].isin(['NON-RESIDENTIAL DEVELOPMENT'])
var = 'is_nonresidential'
varname = r"~ ~ Non-residential development (\%)"
results = get_descriptive_stats_boolean(var)
tbl += rf"{varname} & {results[0]:.3f} & {results[1]:.3f} & {results[2]:.3f} & $\chi^2 = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

df['is_other_casetype'] = (~df['is_residential']) & (~df['is_mixed_use']) & (~df['is_nonresidential'])
var = 'is_other_casetype'
varname = r"~ ~ Other case type (\%)"
results = get_descriptive_stats_boolean(var)
tbl += rf"{varname} & {results[0]:.3f} & {results[1]:.3f} & {results[2]:.3f} & $\chi^2 = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

tbl += r" & & & & & \\" + "\n"
tbl += r"\textit{Physical characteristics} & & & & & \\" + "\n"

df['square_footage'] = pd.to_numeric(df['square_footage'], errors='coerce')
var = 'square_footage'
varname = r"~ ~ Square footage (mean)"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:,.0f} & {results[1]:,.0f} & {results[2]:,.0f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n" 

df['height'] = pd.to_numeric(df['height'], errors='coerce')
var = 'height'
varname = r"~ ~ Height (ft, mean)"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:,.0f} & {results[1]:,.0f} & {results[2]:,.0f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n" 

df['dwelling_units'] = pd.to_numeric(df['dwelling_units'], errors='coerce')
var = 'dwelling_units'
varname = r"~ ~ Dwelling units (mean)"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:,.0f} & {results[1]:,.0f} & {results[2]:,.0f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n" 

df['affordable_units'] = pd.to_numeric(df['affordable_units'], errors='coerce')
var = 'affordable_units'
varname = r"~ ~ Affordable units (mean)"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:,.0f} & {results[1]:,.0f} & {results[2]:,.0f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n" 

df['parking_spaces'] = pd.to_numeric(df['parking_spaces'], errors='coerce')
var = 'parking_spaces'
varname = r"~ ~ Parking spaces (mean)"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:,.0f} & {results[1]:,.0f} & {results[2]:,.0f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n" 

tbl += r" & & & & & \\" + "\n"
tbl += r"\textit{Planning characteristics} & & & & & \\" + "\n"

var = 'sfx_grp_SPR'
varname = r"~ ~ Site plan review (\%)"
results = get_descriptive_stats_boolean(var)
tbl += rf"{varname} & {results[0]:.3f} & {results[1]:.3f} & {results[2]:.3f} & $\chi^2 = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

var = 'sfx_grp_APP'
varname = r"~ ~ Appealed (\%)"
results = get_descriptive_stats_boolean(var)
tbl += rf"{varname} & {results[0]:.3f} & {results[1]:.3f} & {results[2]:.3f} & $\chi^2 = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

var = 'sfx_grp_IP'
varname = r"~ ~ Incentive programs (\%)"
results = get_descriptive_stats_boolean(var)
tbl += rf"{varname} & {results[0]:.3f} & {results[1]:.3f} & {results[2]:.3f} & $\chi^2 = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

var = 'sfx_grp_CUP'
varname = r"~ ~ Conditional use permit (\%)"
results = get_descriptive_stats_boolean(var)
tbl += rf"{varname} & {results[0]:.3f} & {results[1]:.3f} & {results[2]:.3f} & $\chi^2 = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

var = 'sfx_grp_CR'
varname = r"~ ~ Compliance review (\%)"
results = get_descriptive_stats_boolean(var)
tbl += rf"{varname} & {results[0]:.3f} & {results[1]:.3f} & {results[2]:.3f} & $\chi^2 = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

var = 'sfx_grp_VAE'
varname = r"~ ~ Requests zoning change / variance / adjustments (\%)"
results = get_descriptive_stats_boolean(var)
tbl += rf"{varname} & {results[0]:.3f} & {results[1]:.3f} & {results[2]:.3f} & $\chi^2 = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

tbl += r" & & & & & \\" + "\n"
tbl += r"\textit{Hearing characteristics} & & & & & \\" + "\n"

var = 'agenda_order'
varname = r"~ ~ Item agenda order (mean)"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:,.1f} & {results[1]:,.1f} & {results[2]:,.1f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n" 

var = 'num_agenda_items'
varname = r"~ ~ \# Items on agenda (mean)"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:,.1f} & {results[1]:,.1f} & {results[2]:,.1f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n" 


tbl += r" & & & & & \\" + "\n"

var = 'mahalanobis'
varname = r"Atypicality"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:,.2f} & {results[1]:,.2f} & {results[2]:,.2f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n" 

print(header + tbl + footer)

with open(os.path.join(LOCAL_PATH, "tables", "tab_bivariate_descriptives_project_characteristics.tex"), "w", encoding='utf-8') as f:
    f.write(header + tbl + footer)




\begin{table}[H]
\centering
\caption{Descriptive Statistics by Motion Outcome---Project Characteristics}
\vspace{0.2cm}
\label{tab_bivariate_descriptives_project_characteristics}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lrrrll} \toprule
 Variable & \makecell{Approved \\ ($n = 354$)} & \makecell{Approved w/ \\ conditions \\ ($n = 328$)} & \makecell{Delayed / \\ denied \\ ($n = 136$)} & \makecell{Test stat} & \makecell{p-value}  \\ \midrule
 & & & & & \\
\textit{Case type} & & & & & \\
~ ~ Residential development (\%) & 0.373 & 0.265 & 0.243 & $\chi^2 = 12.53$ & $0.002^{***}$ \\
~ ~ Mixed use development (\%) & 0.299 & 0.335 & 0.338 & $\chi^2 = 1.25$ & $0.535^{}$ \\
~ ~ Non-residential development (\%) & 0.096 & 0.207 & 0.191 & $\chi^2 = 17.46$ & $0.000^{***}$ \\
~ ~ Other case type (\%) & 0.232 & 0.192 & 0.228 & $\chi^2 = 1.74$ & $0.420^{}$ \\
 & & & & & \\
\textit{Physical characteristics} & & & & & \\
~ ~ Square footage (mean) & 131,658 & 231,355

In [27]:
for col in list(df.columns):
    print(col)

date
year
item_no
sub_item_no
item_order_changed
taken_off_consent_calendar
multiple_votes
mover
seconder
ayes
noes
abstentions
absences
recusals
vote_result
appeal_result
project_result
minutes_summary
is_consent_calendar
agenda_text
title
address
council_district
council_member
last_day_to_act
referenced_laws
is_appeal
summary_of_appeal
project_type
new_or_existing
square_footage
dwelling_units
affordable_units
height
parking_spaces
requested_actions
agenda_summary
is_casenum
num_ayes
num_noes
unanimous
n_docs
n_support
n_oppose
agenda_order
num_agenda_items
cluster
mahalanobis
euclidean
cosine
sfx_SPR
sfx_1A
sfx_DB
sfx_CU
sfx_HCA
sfx_TOC
sfx_HD
sfx_SPP
sfx_MCUP
sfx_ZC
sfx_GPA
sfx_VZC
sfx_WDI
sfx_VHCA
sfx_CA
sfx_CUB
sfx_PHP
sfx_ZV
sfx_TDR
sfx_SP
sfx_ZAA
sfx_DRB
sfx_GPAJ
sfx_CDP
sfx_SN
sfx_VCU
sfx_VZCJ
sfx_SIP
sfx_CU3
sfx_BL
sfx_ZAD
sfx_CUX
sfx_CDO
sfx_CN
sfx_MEL
sfx_PA1
sfx_ZCJ
sfx_DD
sfx_PR
sfx_DA
sfx_SPE
sfx_BSA
sfx_SPPC
sfx_RDP
sfx_MSC
sfx_F
sfx_M3
sfx_MSP
sfx_PSH
sfx_CPIOA
sfx_CP